# Part 1: PCA — Dimensionality Reduction
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Tuesday — Visualising 10-Dimensional Data on a 2D Screen

Sarah has 10 customer features. She wants to look at the data on a chart. Charts have 2 dimensions. So she has to compress 10 dimensions into 2 — without losing the signal.

PCA does exactly that.

**By the end of this notebook you will be able to:**
- Apply PCA to standardised numerical features
- Read a scree plot and decide how many components to keep
- Project the data into 2D and look at the result
- Interpret what each principal component "means" by looking at its loadings

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — PCA ready")

## Step 1 — Load + preprocess

Same preprocessing as L03/L04 (median impute + standard scale + one-hot encode). **Critical:** PCA is variance-based, so we MUST scale before fitting.

In [ ]:
df = pd.read_csv("data/northstar_customers.csv")
features = df.drop(columns=["customer_id", "churned"])  # unsupervised — no labels

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

X_processed = preprocessor.fit_transform(features)
feature_names_processed = preprocessor.get_feature_names_out()

print(f"Original features:    {features.shape[1]}  ({len(numeric_features)} numeric + {len(categorical_features)} categorical)")
print(f"After preprocessing:  {X_processed.shape[1]} columns  ({len(numeric_features)} scaled num + one-hot expansions)")

## Step 2 — Fit PCA with ALL components

We start by fitting PCA WITHOUT reducing — we just want to see the variance distribution across all components.

In [ ]:
pca_full = PCA()  # no n_components → keeps everything
pca_full.fit(X_processed)

variance_df = pd.DataFrame({
    "component":              [f"PC{i+1}" for i in range(len(pca_full.explained_variance_ratio_))],
    "variance_explained":     pca_full.explained_variance_ratio_,
    "cumulative_variance":    np.cumsum(pca_full.explained_variance_ratio_),
})
print(variance_df.head(10).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## ⏸️ Pause and Predict

Before plotting, predict:
- How many PCs would you need to capture 80% of total variance?
- What does it mean if the FIRST PC explains 0.20 of variance vs 0.50 of variance?

> *Sample:* If PC1 explains 0.50, the data has one strong "axis" — most variation lives in one direction. If PC1 explains 0.20, the variation is spread across many directions, meaning PCA doesn't compress much. Look at the variance_df above before scrolling.

## Step 3 — The scree plot

Plot variance per component. Look for the "elbow" — the point where adding more components doesn't help much.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: individual variance per PC
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_,
            color="steelblue", edgecolor="white")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title("Variance per component (the scree plot)")

# Right: cumulative variance
axes[1].plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
             np.cumsum(pca_full.explained_variance_ratio_),
             "o-", linewidth=2, color="coral", markersize=8)
axes[1].axhline(0.8, color="gray", linestyle="--", label="80% threshold")
axes[1].axhline(0.9, color="gray", linestyle=":", label="90% threshold")
axes[1].set_xlabel("Number of components kept")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title("Cumulative variance")
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

n_for_80 = (np.cumsum(pca_full.explained_variance_ratio_) >= 0.80).argmax() + 1
n_for_90 = (np.cumsum(pca_full.explained_variance_ratio_) >= 0.90).argmax() + 1
print(f"Components needed for 80% variance: {n_for_80}")
print(f"Components needed for 90% variance: {n_for_90}")

### 💡 What you should notice

- The first few PCs capture most of the variance. After that the curve flattens.
- For VISUALISATION, we'll use the first 2 PCs (because we can plot in 2D). For PREPROCESSING, we'd keep more (typically enough for 80–90% variance).
- The fact that we need quite a few components for 80% tells us the features ARE mostly independent — confirming the correlation finding from NB 01.

## Step 4 — Project to 2D and plot

Now reduce to 2 components and plot the customers in PC1 × PC2 space. This is the view we'd put in front of Marcus on Friday.

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_processed)

print(f"Variance captured by 2 components: {pca_2d.explained_variance_ratio_.sum():.1%}")
print()

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.3, s=10, color="steelblue")
ax.set_xlabel(f"PC1  ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2  ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("NorthStar customers in 2D PCA space")
ax.axvline(0, color="gray", linewidth=0.5)
ax.axhline(0, color="gray", linewidth=0.5)
plt.tight_layout()
plt.show()

### 💡 What you should notice

- The customers form a **single cloud** — no obvious clusters in the eyeball view.
- This is normal for real datasets. K-Means (tomorrow) will find centroids that group the cloud anyway — clusters often AREN'T visually obvious.
- The data has structure; PCA preserves the most variance-rich part of it. Whether that structure is *interpretable* depends on what's loading onto each PC — let's check.

## Step 5 — Interpret PC1 and PC2 by looking at the loadings

`pca_2d.components_` tells us how each original feature contributes to each PC. Large absolute values = strong contribution. Sign tells direction.

In [ ]:
loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=["PC1", "PC2"],
    index=feature_names_processed,
)

# Clean up the feature names
loadings.index = [name.split("__")[-1] for name in loadings.index]

# Sort by absolute PC1 loading
print("Features sorted by |PC1 loading|:")
print(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(8).round(3).to_string())

# Heatmap visualisation
fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(loadings.round(3), annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, cbar_kws={"label": "Loading"}, ax=ax)
ax.set_title("PCA loadings — how original features compose PC1 and PC2")
plt.tight_layout()
plt.show()

### 💡 What you should notice

Look at the heatmap and try to **name** each PC:

- **PC1** has strong positive loadings on features like... (look at the top contributors). Customers high on PC1 are *what kind*?
- **PC2** has its own pattern. What is PC2 about?

For NorthStar's customers, common interpretations:
- One PC often captures *engagement / activity* (tenure, purchases, spending all positive together)
- Another captures *contrast between satisfaction and dissatisfaction* (returns, support tickets vs reviews)

Sarah's job on Friday is to NAME these axes for Marcus.

## ✅ Section Summary

| Step | Output |
|---|---|
| **Preprocessed** | 8 numerics scaled + 9 one-hot columns = 17-dimensional input |
| **Fit PCA on all components** | Variance distribution visible in scree plot |
| **Components for 80% variance** | Typically 6–8 — features are mostly independent |
| **First 2 PCs together** | ~22% of total variance (modest — features are nearly independent, so PCA can't compress hard) |
| **2D plot** | A single smooth cloud; no obvious clusters visually |
| **Loadings** | Tell us what each PC "means" in original-feature terms |

**Key insight:**
> PCA doesn't FIND clusters — it just gives you the best 2D view of the data. Whether clusters EXIST in that view is K-Means's job (tomorrow). What PCA gives you is the *coordinate system* in which clustering makes the most sense.

> **Don't interpret 22% as "PCA failed".** It's the *correct* answer when features carry mostly independent information. For visualisation a single 2D cloud is still useful; for preprocessing you'd keep 6-8 components (the 80%-variance threshold).

---
**Up next → Part 2:** Wednesday — K-Means clustering. Find natural groups + decide K + interpret each cluster.
Open `03_kmeans.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — How many components do you NEED for downstream modelling?

If you're using PCA for VISUALISATION, you use 2 components (because 2D). If you're using it for PREPROCESSING (feeding the result into a downstream model), keep enough for 90–95% variance. Let's compare downstream model performance with different PCA dimensions.

(We'll use the L03 churn target just for this Extension demo — but it's no longer the main lesson goal.)

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

y = df["churned"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
records = []

# Baseline: no PCA, all 17 features
baseline_f1 = cross_val_score(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    X_processed, y, cv=cv, scoring="f1", n_jobs=-1,
).mean()

for n_pc in [2, 5, 10, 15]:
    if n_pc > X_processed.shape[1]: continue
    pca = PCA(n_components=n_pc)
    X_pca = pca.fit_transform(X_processed)
    f1 = cross_val_score(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        X_pca, y, cv=cv, scoring="f1", n_jobs=-1,
    ).mean()
    var = pca.explained_variance_ratio_.sum()
    records.append((n_pc, var, f1))

df_demo = pd.DataFrame(records, columns=["n_components", "var_explained", "downstream_cv_f1"])
print(f"Baseline (no PCA, all 17 features): CV F1 = {baseline_f1:.3f}")
print()
print(df_demo.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### 💡 What this tells us

- **Reducing to 2 components destroys signal** — much worse F1 than baseline.
- **Going to 10-15 components recovers most of the signal**.
- For modelling, PCA is rarely the right choice on tabular data — it adds preprocessing complexity without gain. It IS useful for very high-dim data (text embeddings, image features) where the original dimensions are noisy.

## Extension 2 — Why scaling matters

If you skip the scaler, the feature with the biggest natural range dominates PC1. Let's prove it.

In [ ]:
# PCA on UNSCALED numerical features
features_num_only = features[numeric_features].fillna(features[numeric_features].median())

pca_unscaled = PCA(n_components=2)
pca_unscaled.fit(features_num_only)

loadings_unscaled = pd.DataFrame(
    pca_unscaled.components_.T,
    columns=["PC1", "PC2"],
    index=numeric_features,
)
print("PCA on UNSCALED data — PC1 loadings:")
print(loadings_unscaled["PC1"].abs().sort_values(ascending=False).head(4).round(3))
print()
print("→ avg_monthly_spend_gbp will dominate PC1 because its range (0-500) is the biggest.")
print("→ This isn't 'most informative' — it's just biggest. Always scale first.")

## Extension 3 — Compare PCA to t-SNE (non-linear)

PCA finds the best LINEAR projection. For non-linear structure, methods like t-SNE or UMAP often look better. They're slower and harder to interpret, but the 2D plots can reveal clusters PCA misses.

In [ ]:
from sklearn.manifold import TSNE

# Use the same processed data; subsample for speed (t-SNE is O(n²))
n_sample = 2000
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_processed), size=n_sample, replace=False)
X_sample = X_processed[sample_idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_sample)

# Side-by-side
X_pca_sample = pca_2d.transform(X_sample)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(X_pca_sample[:, 0], X_pca_sample[:, 1], alpha=0.4, s=12, color="steelblue")
axes[0].set_title(f"PCA — first 2 components ({n_sample} customers)")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.4, s=12, color="coral")
axes[1].set_title(f"t-SNE 2D embedding ({n_sample} customers)")
axes[1].set_xlabel("Dim 1"); axes[1].set_ylabel("Dim 2")

plt.tight_layout()
plt.show()

print("PCA: linear, fast, interpretable. Use for preprocessing + first-look viz.")
print("t-SNE: non-linear, slow, prettier viz. Use when PCA's 2D view looks featureless.")